# Experiment 31 | YOLO11n-seg — Independent training seeds (GPU)

**Computing environment:** Google Colab **NVIDIA GPU T4** (or equivalent CUDA GPU). **Do not run training on CPU.**

**Scientific objective.** Complete seeds **2027** and **2028** while **preserving the completed seed 2026** from Experiment 03. This notebook does not evaluate, tune, or inspect predictions on the test set. The date-disjoint split is frozen. This is an assessment of initialization/training variability under the **same YOLO11n-seg recipe**, not proof of balanced training cost against Mask R-CNN.

**Run order:** select GPU in Colab → *Runtime > Run all*. Approve Drive access. Each completed seed is skipped on subsequent runs. Checkpoints can be resumed without mixing seed folders.

**Output:** model weights remain in `03_Resultados/06_Entrenamiento/YOLO11n_multisemilla_v1`; summary Excel, JSON, and English publication-ready figures are saved to `03_Resultados`; a lightweight ZIP is copied to `05_Exportaciones_ZIP` and downloaded automatically. **The ZIP does not include model weights.**

In [ ]:
%pip -q install "ultralytics==8.4.182" "pyyaml>=6,<7" "pandas>=2,<3" "openpyxl>=3.1,<4" "matplotlib>=3.7"

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, sys, platform, zipfile, time, random
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import yaml, torch, ultralytics
from ultralytics import YOLO
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA unavailable. In Colab, select Runtime > Change runtime type > T4 GPU. Training on CPU is not supported.")
assert ultralytics.__version__ == "8.4.182", f"Unexpected Ultralytics version {ultralytics.__version__}"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| ultralytics", ultralytics.__version__)
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
if not ROOT.is_dir():
    raise FileNotFoundError(f"Project root not found: {ROOT}. Check the Google account mounted in Colab.")
SPLIT=ROOT/'01_Datasets'/'FaroPigSeg'/'split_fecha_v1'
YAML=SPLIT/'data_fecha.yaml'
MANIFEST=SPLIT/'particion_fecha_manifest.json'
RES=ROOT/'03_Resultados'
ORIGINAL=RES/'06_Entrenamiento'/'YOLO11n_fecha_v1'
TRAIN=RES/'06_Entrenamiento'/'YOLO11n_multisemilla_v1'
REPORT=RES/'05_Reportes'/'CERDOS_31_YOLO11n_Semillas'
FIG=RES/'04_Figuras'/'CERDOS_31_YOLO11n_Semillas'
METRICS=RES/'03_Metricas'
EXPORT=ROOT/'05_Exportaciones_ZIP'
for p in (TRAIN,REPORT,FIG,METRICS,EXPORT):p.mkdir(parents=True,exist_ok=True)

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(2**20),b''): h.update(b)
    return h.hexdigest()

def savejson(path,obj):
    Path(path).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

print("Root:",ROOT)

In [ ]:
# Preflight: frozen experimental split; no test predictions are ever loaded.
assert YAML.is_file() and MANIFEST.is_file(), "Missing frozen split artifacts: execute notebook 02 before training."
cfg=yaml.safe_load(YAML.read_text(encoding='utf-8'))
part=json.loads(MANIFEST.read_text(encoding='utf-8'))
EXPECTED={'train':949,'val':196,'test':373}
DATE_COUNTS={'train':16,'val':4,'test':5}
PATHS={};DATES={}
for split in ('train','val','test'):
    p=Path(cfg[split]); assert p.is_file(), f"Missing split list {split}: {p}"
    lines=[Path(v.strip()) for v in p.read_text(encoding='utf-8').splitlines() if v.strip()]
    assert len(lines)==EXPECTED[split],f"Unexpected {split} count {len(lines)}"
    assert len(lines)==len({str(x) for x in lines}),f"Duplicated paths in {split}"
    PATHS[split]=lines;DATES[split]={p.name[:10] for p in lines}
    assert len(DATES[split])==DATE_COUNTS[split],f"Unexpected distinct dates in {split}"
    for day in DATES[split]:
        assert part['asignaciones_fecha'][day]==split, f"Manifest disagrees with {split}: {day}"
for a,b in (('train','val'),('train','test'),('val','test')):
    assert DATES[a].isdisjoint(DATES[b]),f"Date leakage between {a} and {b}"
    assert {str(p) for p in PATHS[a]}.isdisjoint(str(p) for p in PATHS[b]),f"Image leakage between {a} and {b}"
for split in ('train','val'):
    for p in PATHS[split]:
        assert p.is_file(),f"Missing training/validation image: {p}"
        lab=Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')
        assert lab.is_file(),f"Missing training/validation annotation: {lab}"
FREEZE={'yaml_sha256':sha256(YAML), 'manifest_sha256':sha256(MANIFEST)}
for split in ('train','val','test'):
    FREEZE[f'{split}_txt_sha256']=sha256(Path(cfg[split]))
print("Split checked:", {k:(len(PATHS[k]),len(DATES[k])) for k in PATHS})
print("YAML SHA-256:",FREEZE['yaml_sha256'])

In [ ]:
# Verify original 2026 seed exists and was trained with the intended recipe.
ORIG_BEST=ORIGINAL/'weights'/'best.pt'
ORIG_ARGS=ORIGINAL/'args.yaml'
ORIG_HIST=ORIGINAL/'results.csv'
assert ORIG_BEST.is_file() and ORIG_ARGS.is_file() and ORIG_HIST.is_file(),     "Original seed 2026 is missing a required file; do NOT restart or overwrite it."
old=yaml.safe_load(ORIG_ARGS.read_text(encoding='utf-8'))
ORIG_RECIPE={'model':'yolo11n-seg.pt','epochs':60,'imgsz':640,'batch':8,'patience':15,
             'workers':2,'seed':2026,'deterministic':True}
for k,v in ORIG_RECIPE.items():
    assert old.get(k)==v, f"Original configuration mismatch: {k}, observed={old.get(k)}, expected={v}"
assert Path(old['data']).resolve()==YAML.resolve(),"Original and new datasets point to different configurations."
assert old.get('optimizer')=='auto',"Original optimizer differs from auto."
print("Original 2026 preserved:",ORIG_BEST)
print("Original train epochs recorded:",len(pd.read_csv(ORIG_HIST)))

In [ ]:
# Matching the Experiment 03 training recipe. The ONLY intended change is the random seed.
SEEDS_TO_RUN=[2027,2028]
MODEL='yolo11n-seg.pt'
EPOCHS=60; IMGSZ=640; BATCH=8; PATIENCE=15; WORKERS=2
assert SEEDS_TO_RUN==[2027,2028]
RECIPE={'model':MODEL,'epochs':EPOCHS,'imgsz':IMGSZ,'batch':BATCH,'patience':PATIENCE,
        'workers':WORKERS,'deterministic':True,'optimizer':'auto',
        'amp':True,'close_mosaic':10,'save_period':10,'ultralytics':'8.4.182',
        'frozen_split':FREEZE}
PROTOCOL={'experiment':31,'training_model':MODEL,'original_seed':2026,
          'new_seeds':SEEDS_TO_RUN,'recipe':RECIPE,'original_weights':str(ORIG_BEST),
          'original_sha256':sha256(ORIG_BEST),'device':torch.cuda.get_device_name(0),
          'torch':torch.__version__,'python':platform.python_version(),
          'no_test_model_selection':True,
          'limitations':'Does not equalize the YOLO and Mask R-CNN training budgets.'}
savejson(REPORT/'CERDOS_31_Protocol.json',PROTOCOL)

def run_one(seed):
    out=TRAIN/f'seed_{seed}'
    weights=out/'weights'; best=weights/'best.pt';last=weights/'last.pt'
    control=out/'CERDOS_31_Config.json'; done=out/'CERDOS_31_Completed.json'
    required={'seed':seed,'recipe':RECIPE}
    if done.is_file():
        assert best.is_file(),f"Completion marker present without best.pt: {out}"
        olddone=json.loads(done.read_text(encoding='utf-8'))
        assert olddone['required']==required, f"Completed run configuration changed: {out}"
        assert olddone['weights_sha256']==sha256(best), f"Completed weights changed: {best}"
        print("Already completed (skipped):",out.name)
        return out
    if control.is_file():
        oldconfig=json.loads(control.read_text(encoding='utf-8'))
        assert oldconfig==required,f"Partial run configuration mismatch: {out}"
    else:
        if best.exists() or last.exists() or (out/'results.csv').exists():
            raise RuntimeError(f"Found old training output without control metadata: {out}. Audit before continuing.")
        out.mkdir(parents=True,exist_ok=True)
        savejson(control,required)
    started=time.monotonic()
    if last.is_file():
        print("Resuming checkpoint for",seed)
        YOLO(str(last)).train(resume=True)
    else:
        print("Starting new seed from the same pretrained weights:",seed)
        model=YOLO(MODEL)
        assert model.task=='segment',"The base model is not instance segmentation."
        model.train(data=str(YAML),epochs=EPOCHS,imgsz=IMGSZ,batch=BATCH,
                    device=0,seed=seed,deterministic=True,patience=PATIENCE,
                    workers=WORKERS,project=str(TRAIN),name=f'seed_{seed}',
                    exist_ok=True,save=True,save_period=10,plots=True,
                    optimizer='auto',amp=True,close_mosaic=10)
    if not best.is_file() or not (out/'results.csv').is_file():
        raise RuntimeError(f"Training did not produce best.pt and results.csv: {out}")
    hist=pd.read_csv(out/'results.csv');hist.columns=[x.strip() for x in hist.columns]
    assert len(hist)>0
    metric='metrics/mAP50-95(M)'
    assert metric in hist,f"Validation segmentation metric absent from {out/'results.csv'}"
    ap=pd.to_numeric(hist[metric],errors='coerce')
    assert ap.notna().any(),f"No validation AP recorded: {out}"
    record={'required':required,'completed_utc':datetime.now(timezone.utc).isoformat(),
            'weights_sha256':sha256(best), 'args_sha256':sha256(out/'args.yaml'),
            'actual_epochs':len(hist),'best_validation_mask_AP':float(ap.max()),
            'best_epoch':int(pd.to_numeric(hist['epoch']).iloc[ap.idxmax()]),
            'runtime_s_this_launch':round(time.monotonic()-started,2),
            'test_not_evaluated':True}
    savejson(done,record)
    print("Completed",seed,"| best val mask AP",round(record['best_validation_mask_AP'],4))
    return out

RUNS=[run_one(s) for s in SEEDS_TO_RUN]

In [ ]:
# Summarize the original run (2026) and the two newly trained runs (2027/2028).
sources={2026:ORIGINAL,2027:TRAIN/'seed_2027',2028:TRAIN/'seed_2028'}
summary=[]; allrows=[]
for seed,folder in sources.items():
    p=folder/'results.csv'
    assert p.is_file(),f"Training history missing: {p}"
    h=pd.read_csv(p);h.columns=[c.strip() for c in h.columns]
    metric='metrics/mAP50-95(M)'; assert metric in h and 'epoch' in h
    v=pd.to_numeric(h[metric],errors='coerce'); assert v.notna().any()
    i=v.idxmax();best=folder/'weights'/'best.pt'
    assert best.is_file(),f"Missing best weights: {best}"
    summary.append({'seed':seed,'status':'original' if seed==2026 else 'new',
                    'best_validation_mask_AP':float(v[i]),
                    'best_epoch':int(h.loc[i,'epoch']),
                    'epochs_recorded':len(h),'weights_sha256':sha256(best),
                    'weights_path':str(best)})
    h.insert(0,'seed',seed)
    allrows.append(h)
summary_df=pd.DataFrame(summary).sort_values('seed').reset_index(drop=True)
history_df=pd.concat(allrows,ignore_index=True)
print(summary_df[['seed','status','best_validation_mask_AP','best_epoch','epochs_recorded']].to_string(index=False))
EXCEL=METRICS/'CERDOS_31_YOLO11n_MultiSeed_Validation.xlsx'
with pd.ExcelWriter(EXCEL,engine='openpyxl') as w:
    summary_df.to_excel(w,sheet_name='TrainingSummary',index=False)
    history_df.to_excel(w,sheet_name='ValidationHistory',index=False)
print('Saved:',EXCEL)

In [ ]:
# Scientific figures: English only, PNG 600 DPI + vector PDF and SVG.
plt.rcParams.update({'font.size':10, 'axes.titlesize':12,'axes.labelsize':11,'savefig.bbox':'tight'})
fig,ax=plt.subplots(figsize=(6.7,4.0))
for seed,df in history_df.groupby('seed'):
    ax.plot(df['epoch'],df['metrics/mAP50-95(M)'],lw=1.9,marker=None,label=f'Seed {seed}')
ax.set(xlabel='Training epoch',ylabel='Validation mask AP (IoU 0.50–0.95)',
       title='YOLO11n-seg: validation performance across random seeds')
ax.legend(frameon=False)
ax.grid(alpha=.22)
for extension in ('png','pdf','svg'):
    fig.savefig(FIG/f'CERDOS_31_Validation_AP_Curves.{extension}',dpi=600)
plt.show();plt.close(fig)

fig,ax=plt.subplots(figsize=(6.4,3.8))
seeds=summary_df['seed'].astype(str)
vals=summary_df['best_validation_mask_AP']
ax.bar(seeds,vals,width=.54,color='#446C94')
for i,v in enumerate(vals):ax.text(i,v+.008,f'{v:.3f}',ha='center',fontsize=10)
ax.set(xlabel='Random seed',ylabel='Best validation mask AP (IoU 0.50–0.95)',
       title='YOLO11n-seg: best validation checkpoint by seed',ylim=(0,min(1,float(vals.max())+.065)))
ax.grid(axis='y',alpha=.2)
for extension in ('png','pdf','svg'):
    fig.savefig(FIG/f'CERDOS_31_Best_Validation_AP.{extension}',dpi=600)
plt.show();plt.close(fig)

In [ ]:
# Traceability report and lightweight ZIP export; model weights stay on Drive.
result={'experiment':31,'created_utc':datetime.now(timezone.utc).isoformat(),
        'model':'YOLO11n-seg','seeds':[2026,2027,2028],
        'source_of_seed_2026':'Experiment 03, unchanged',
        'summary':summary_df.to_dict(orient='records'),
        'protocol_path':str(REPORT/'CERDOS_31_Protocol.json'),
        'test_evaluated':False,'note':'Validation-only training summary; test comparison is not performed here.'}
OUTJSON=REPORT/'CERDOS_31_Training_Report.json'
savejson(OUTJSON,result)
ZIP=EXPORT/'CERDOS_31_YOLO11n_Semillas_Resultados.zip'
files=[EXCEL,OUTJSON,REPORT/'CERDOS_31_Protocol.json']+list(FIG.glob('CERDOS_31_*'))
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for file in files:
        if not file.is_file():raise FileNotFoundError(file)
        z.write(file,arcname=file.name)
print('Artifacts saved to project Drive.')
print('ZIP:',ZIP,'| size:',ZIP.stat().st_size,'bytes')
from google.colab import files as colabfiles
colabfiles.download(str(ZIP))